<a href="https://colab.research.google.com/github/neysazefanya/Eksplorasi-Probabilitas-Python-Neysa/blob/main/Tugas_Eksplorasi_Probabilitas_Neysa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas Eksplorasi dan Implementasi Python: Probabilitas dan Distribusi Probabilitas

**Nama:** Neysa Zefanya  **NIM:** 2455301232  **Mata Kuliah:** Probabilitas dan Statistika


## 0. Eksplorasi Library Python

| Library | Fungsi utama | Kegunaan | Contoh | Dipakai pada soal |
|---|---|---|---|---|
| `math` | `comb`, `perm`, `factorial`, `exp` | Kombinatorik, rumus manual | `math.comb(5,2)` | A12, A13, B1, B7 |
| `itertools` | `product`, `permutations`, `combinations` | Enumerasi ruang sampel | `itertools.product("01", repeat=2)` | A2, A3, A6, A7, A12, A13 |
| `fractions` | `Fraction` | Peluang berbentuk pecahan eksak | `Fraction(3,12)` | A3, A7, A13 |
| `random` | `choice`, `sample`, `seed` | Percobaan acak sederhana | `random.sample(range(1,50),3)` | demo, seed |
| `statistics` | `mean`, `pvariance`, `mode`, `NormalDist` | Statistik deskriptif dan normal bawaan | `statistics.mean(data)` | A1, B11 |
| `numpy` | `random.*`, `cumsum`, `linalg` | Simulasi Monte Carlo, matriks | `np.random.binomial(10,.5,5)` | seluruh simulasi (A5, A14–A19, B3, B9, B13) |
| `scipy.stats` | `pmf/pdf`, `cdf`, `sf`, `ppf`, `rvs` | Hipergeometrik, binomial, Poisson, normal | `stats.binom(10,.5).pmf(5)` | Bagian B |
| `sympy` | `symbols`, `Rational`, `summation`, `integrate`, `solve` | Rumus simbolik / eksak | `sp.integrate(2*sp.exp(-2*x),(x,0,1))` | A18, B1, B4, B7, B10 |
| `pandas` | `DataFrame`, `value_counts` | Tabel frekuensi & ringkasan | `pd.Series(d).value_counts(normalize=True)` | A1, A9, B4, B14 |
| `matplotlib` | `bar`, `plot`, `fill_between` | Visualisasi pmf/pdf/simulasi | `plt.bar(x, d.pmf(x))` | hampir semua soal |
| `seaborn` | `barplot`, `histplot`, `heatmap` | Grafik statistik yang lebih ringkas | `sns.histplot(d, kde=True)` | A1, A19, B9, B13 |
| `statsmodels` | `proportion_confint`, `qqplot` | Interval kepercayaan simulasi, uji kenormalan | `proportion_confint(334,1000)` | A5, A15, B3, B13 |

In [ ]:
import math, itertools, random, statistics, os
from fractions import Fraction
import numpy as np
import pandas as pd
import sympy as sp
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.proportion import proportion_confint
from scipy import stats

np.random.seed(42); random.seed(42)
os.makedirs("grafik", exist_ok=True)

def simpan(nama):
    plt.tight_layout()
    plt.savefig(f"grafik/{nama}.png", dpi=130)
    plt.show()          # tampil di Colab sekaligus tersimpan di folder grafik/
N = 500_000      # banyak percobaan simulasi

Demonstrasi tiap library:

In [ ]:
# --- math: perhitungan dasar kombinatorik ---
print("math    :", math.comb(5, 2), math.perm(5, 2), math.factorial(5), round(math.exp(-2), 4))
# --- itertools: enumerasi ruang sampel ---
print("product :", list(itertools.product("01", repeat=2)))
print("combin. :", list(itertools.combinations("ABC", 2)))
# --- fractions: probabilitas eksak ---
print("Fraction:", Fraction(3, 12), "|", Fraction(1, 3) + Fraction(1, 6))
# --- random: percobaan acak sederhana ---
print("random  :", random.choice(["ACK", "NACK"]), random.sample(range(1, 50), 3))
# --- statistics: statistik deskriptif bawaan Python ---
data = [2, 3, 3, 5, 7]
print("stat    :", statistics.mean(data), statistics.variance(data), statistics.mode(data))
# --- numpy: simulasi vektor ---
print("numpy   :", np.random.binomial(10, 0.5, 5), np.random.randint(0, 256, 5))
# --- scipy.stats: objek distribusi (pmf, cdf, ppf, rvs, mean, var) ---
d = stats.binom(10, 0.5)
print("scipy   :", round(d.pmf(5), 4), round(d.cdf(5), 4), d.ppf(0.9), d.mean(), d.var())
# --- sympy: simbolik / eksak ---
x = sp.symbols("x")
print("sympy   :", sp.integrate(2*sp.exp(-2*x), (x, 0, 1)), "|", sp.Rational(1, 3) + sp.Rational(1, 6))
# --- pandas: tabel frekuensi ---
print("pandas  :", pd.Series(np.random.choice([200, 404, 500], 100, p=[0.7, 0.2, 0.1])).value_counts(normalize=True).sort_index().round(2).to_dict())
# --- statsmodels: interval kepercayaan hasil simulasi ---
from statsmodels.stats.proportion import proportion_confint
print("statsm. :", proportion_confint(count=334, nobs=1000, alpha=0.05, method="wilson"))
# --- matplotlib + seaborn: visualisasi ---
sns.histplot(np.random.normal(0, 1, 2000), kde=True, stat="density")
plt.title("seaborn.histplot + kde"); simpan("lib_demo")

---
# BAGIAN A: PROBABILITAS

## 1. Probabilitas dan frekuensi relatif (empiris)

### A1. Probabilitas empiris dari tabel frekuensi: lama downtime server (Modul 04)

Lama pemulihan (downtime, menit ke-1 s.d. ke-10) dari 30 insiden server dicatat pada tabel frekuensi di kode. Hitung frekuensi relatif (probabilitas empiris), nilai harapan lama downtime, variansnya, dan P(downtime ≤ 3 menit).

In [ ]:
menit = np.arange(1, 11)
frek = np.array([2, 3, 8, 4, 5, 3, 2, 0, 1, 2])          # total 30 insiden
df_down = pd.DataFrame({"menit": menit, "frekuensi": frek})
df_down["frek_relatif"] = (df_down.frekuensi / df_down.frekuensi.sum()).round(2)
print(df_down.to_string(index=False), "\nTotal:", frek.sum())
p_emp = frek / frek.sum()
E = (menit*p_emp).sum(); V = ((menit-E)**2*p_emp).sum()
print("E[downtime] =", round(E, 2), "menit | Var =", round(V, 2), "| SD =", round(math.sqrt(V), 2))
print("P(downtime <= 3 menit) =", round(p_emp[:3].sum(), 4), "(manual 13/30 =", round(13/30, 4), ")")
print("Modus =", menit[frek.argmax()], "menit")
data = np.repeat(menit, frek).tolist()                      # data mentah 30 insiden
print("statistics: mean =", round(statistics.mean(data), 2), "| pvariance =", round(statistics.pvariance(data), 2), "| median =", statistics.median(data), "| mode =", statistics.mode(data))
sns.barplot(x=menit, y=p_emp, color="tab:orange"); plt.plot(menit-1, p_emp, "o-", c="brown")
plt.xlabel("Lama downtime (menit)"); plt.ylabel("Frekuensi relatif")
plt.title("Downtime server (30 insiden)"); simpan("A1_downtime")

**Interpretasi:** Rata-rata downtime ≈ 4,47 menit; hanya 43,3% insiden pulih dalam ≤ 3 menit. `statistics` (mean, mode) memberi hasil sama.

## 2. Ruang sampel dan kejadian

### A2. Ruang sampel dan kejadian

Tiga server diperiksa; tiap server boleh **y** (perlu patch/diperbarui) atau **n** (tidak). Tentukan ruang sampel, lalu peluang kejadian A = paling sedikit 2 server perlu patch, B = tidak ada yang perlu patch, C = tepat 2 server perlu patch.

In [ ]:
S1 = list(itertools.product("yn", repeat=3))
print("Ruang sampel:", len(S1), "elemen"); print(S1)
A_ = [s for s in S1 if s.count("y") >= 2]
B_ = [s for s in S1 if s.count("y") == 0]
C_ = [s for s in S1 if s.count("y") == 2]
for nama, e in zip("ABC", [A_, B_, C_]):
    print(f"Event {nama}: {len(e)} elemen -> P = {len(e)}/{len(S1)} = {len(e)/len(S1):.3f}")
print("(asumsi: 8 titik sampel sama mungkin)")

**Interpretasi:** P(A)=0,5; P(B)=0,125; P(C)=0,375 (asumsi 8 titik sampel sama mungkin).

### A3. Operasi kejadian: gabungan, irisan, komplemen (data 4-bit)

Sebuah data 4-bit dibangkitkan acak (tiap bit 0/1 sama mungkin, 16 kemungkinan). A: banyak bit 1 genap (parity genap); B: banyak bit 1 ≥ 3. Hitung P(A), P(B), P(A∪B), P(A∩B), P(Aᶜ) dan buktikan P(A∪B) = P(A) + P(B) − P(A∩B).

In [ ]:
S = list(itertools.product("01", repeat=4))
P = lambda E: Fraction(len(E), len(S))
A_ = {s for s in S if s.count("1") % 2 == 0}
B_ = {s for s in S if s.count("1") >= 3}
print("P(A) =", P(A_), "| P(B) =", P(B_))
print("P(A∩B) =", P(A_ & B_), "| P(A∪B) =", P(A_ | B_), "| P(A^c) =", P(set(S) - A_))
print("Aturan penjumlahan:", P(A_) + P(B_) - P(A_ & B_), "=", P(A_ | B_))

**Interpretasi:** P(A)=1/2; P(B)=5/16; P(A∩B)=1/16; P(A∪B)=3/4; P(Aᶜ)=1/2. Aturan penjumlahan terbukti (3/4 = 3/4).

## 3. Gabungan kejadian (aturan penjumlahan)

### A4. Gabungan kejadian dan inklusi-eksklusi: tiga server

Tiga server independen mati dengan peluang 0,05; 0,10; 0,08. Berapa peluang minimal satu server mati? Hitung lewat komplemen, inklusi-eksklusi, dan simulasi.

In [ ]:
p = [0.05, 0.10, 0.08]
manual = 1 - np.prod([1-x for x in p])
ie = sum(p) - (p[0]*p[1] + p[0]*p[2] + p[1]*p[2]) + p[0]*p[1]*p[2]
print(f"Komplemen: {manual:.5f} | Inklusi-eksklusi: {ie:.5f}")
sim = (np.random.rand(1_000_000, 3) < p).any(axis=1).mean()
print(f"Simulasi: {sim:.5f}")
print(f"Batas atas (union bound) sum(p) = {sum(p):.2f}")

**Interpretasi:** P(minimal satu mati)=0,2134; komplemen = inklusi-eksklusi ≈ simulasi.

## 4. Probabilitas bersyarat dan kejadian saling bebas

### A5. Probabilitas bersyarat berantai: overload server dua jam berturut-turut

Diketahui server sedang overload pada jam ini, dan peluang overload pada jam berikutnya bila jam sebelumnya overload adalah 0,444. Berapa peluang overload pada dua jam berikutnya secara berturut-turut? Bandingkan dengan simulasi.

In [ ]:
p_oo = 0.444
print("Manual: 0.444 x 0.444 =", round(p_oo*p_oo, 4))
N = 500_000
jam1 = np.random.rand(N) < p_oo
jam2 = np.random.rand(N) < p_oo
print("Simulasi:", round((jam1 & jam2).mean(), 4))
lo, hi = proportion_confint(int((jam1 & jam2).sum()), N, alpha=0.05, method="wilson")
print(f"statsmodels: CI 95% simulasi = ({lo:.4f}, {hi:.4f}) | manual 0.1971 di dalam CI? {lo <= 0.1971 <= hi}")

**Interpretasi:** Manual 0,1971 ≈ simulasi 0,197. CI 95% simulasi (statsmodels) memuat nilai manual.

### A6. Probabilitas bersyarat: membuka email tanpa pengembalian

Di kotak masuk ada 5 email: 3 spam dan 2 bukan spam. Dua email dibuka satu per satu secara acak tanpa pengembalian. A: email kedua spam; B: email pertama spam. Hitung P(A|B) dan bandingkan dengan P(A).

In [ ]:
S = list(itertools.permutations(range(5), 2))      # email 0,1,2 = spam; 3,4 = bukan spam
spam = {0, 1, 2}
A_ = {s for s in S if s[1] in spam}
B_ = {s for s in S if s[0] in spam}
print("|A∩B| =", len(A_ & B_), "dari", len(S), "hasil")
print("Manual: P(A∩B)=6/20, P(B)=12/20 -> P(A|B) = 1/2 =", round(1/2, 4))
print("Python P(A|B) =", round(len(A_ & B_) / len(B_), 4), "| P(A) tanpa syarat =", round(len(A_)/len(S), 4))

**Interpretasi:** P(A|B)=1/2 < P(A)=0,6, jadi A dan B tidak saling bebas (tanpa pengembalian: bila email pertama spam, sisa spam berkurang).

### A7. Kejadian saling bebas atau tidak (flag paket TCP)

Sebuah paket memiliki 3 flag (SYN, ACK, FIN); tiap flag aktif (1) atau tidak (0) dengan peluang sama, sehingga ada 8 kemungkinan. B: flag SYN aktif. A: banyak flag aktif genap. C: minimal 2 flag aktif. Periksa apakah A dan B saling bebas, serta apakah C dan B saling bebas.

In [ ]:
S = list(itertools.product((0, 1), repeat=3))      # (SYN, ACK, FIN)
P = lambda E: Fraction(len(E), len(S))
B_ = {s for s in S if s[0] == 1}
A_ = {s for s in S if sum(s) % 2 == 0}
C_ = {s for s in S if sum(s) >= 2}
for nama, E in [("A (banyak flag genap)", A_), ("C (minimal 2 flag)", C_)]:
    print(f"{nama}: P(E∩B) = {P(E & B_)} | P(E)P(B) = {P(E)*P(B_)} -> saling bebas? {P(E & B_) == P(E)*P(B_)}")

**Interpretasi:** A dan B saling bebas (1/4 = 1/4); C dan B tidak (3/8 ≠ 1/4).

## 5. Probabilitas total

### A8. Probabilitas total: lonjakan trafik dan server lambat

P(lonjakan trafik) = 0,36. Bila terjadi lonjakan, server melambat (latency tinggi) dengan peluang 0,80; bila tidak ada lonjakan, server melambat dengan peluang 0,25. Berapa peluang server melambat?

In [ ]:
p_H = 0.36; p_G_H = 0.80; p_G_Hc = 0.25      # H = lonjakan trafik, G = server lambat
p_G = p_H*p_G_H + (1-p_H)*p_G_Hc
print("P(lambat) = 0.36*0.80 + 0.64*0.25 =", round(p_G, 4))
lonjakan = np.random.rand(N) < p_H
lambat = np.where(lonjakan, np.random.rand(N) < p_G_H, np.random.rand(N) < p_G_Hc)
print("Simulasi P(lambat) =", round(lambat.mean(), 4))
print("(Catatan: slide menulis 0.82 pada perhitungan, tetapi data soal 0.80; hasil 0.448 hanya cocok untuk 0.80)")

**Interpretasi:** P(server lambat)=0,448, cocok dengan simulasi. Slide menulis 0,82, padahal data soal 0,80.

## 6. Teorema Bayes

### A9. Teorema Bayes: saluran pengiriman

Pesan dikirim lewat 4 saluran R1..R4 dengan peluang pemakaian 0,1; 0,2; 0,3; 0,4. Peluang pesan salah (E) pada tiap saluran: 0,10; 0,15; 0,20; 0,25. Jika sebuah pesan ternyata salah, berapa peluang pesan itu lewat saluran R2?

In [ ]:
P_R = np.array([0.1, 0.2, 0.3, 0.4])
P_E_R = np.array([0.10, 0.15, 0.20, 0.25])
gabungan = P_R * P_E_R
P_E = gabungan.sum()
tabel = pd.DataFrame({"i": [1,2,3,4], "P(Ri)": P_R, "P(E|Ri)": P_E_R,
                      "P(Ri)P(E|Ri)": gabungan.round(3), "P(Ri|E)": (gabungan/P_E).round(2)})
print(tabel.to_string(index=False))
print("P(E) =", round(P_E, 2), "| P(R2|E) =", round(gabungan[1]/P_E, 2), "(slide: 0.15)")
saluran = np.random.choice(4, size=N, p=P_R)
salah = np.random.rand(N) < P_E_R[saluran]
print("Simulasi P(R2|E) =", round((saluran[salah] == 1).mean(), 4))
plt.bar([1,2,3,4], P_R, alpha=.5, label="Prior P(Ri)")
plt.bar([1,2,3,4], gabungan/P_E, alpha=.7, label="Posterior P(Ri|E)")
plt.xlabel("Saluran"); plt.xticks([1,2,3,4]); plt.legend(); plt.title("Prior vs Posterior (Bayes)"); simpan("A5_bayes")

**Interpretasi:** P(R2|E)=0,15 (sesuai slide). Bukti 'pesan salah' menaikkan peluang R4 dari 0,40 ke 0,50.

### A10. Teorema Bayes: filter spam

30% email adalah spam. Kata "gratis" muncul pada 80% spam dan 5% email bukan spam. Jika sebuah email memuat kata "gratis", berapa peluang email itu spam?

In [ ]:
p_spam = 0.30; p_kata_spam = 0.80; p_kata_ham = 0.05
p_kata = p_kata_spam*p_spam + p_kata_ham*(1-p_spam)
p_spam_kata = p_kata_spam*p_spam / p_kata
print(f"P(\"gratis\") = {p_kata:.4f}")
print(f"P(spam | \"gratis\") = {p_spam_kata:.4f}")
N = 500_000
spam = np.random.rand(N) < p_spam
kata = np.where(spam, np.random.rand(N) < p_kata_spam, np.random.rand(N) < p_kata_ham)
print(f"Simulasi P(spam | gratis) = {spam[kata].mean():.4f}")

**Interpretasi:** P(spam|'gratis')=0,8727, naik dari prior 0,30; simulasi cocok.

### A11. Teorema Bayes dan base rate: Intrusion Detection System

Sebuah IDS mendeteksi serangan dengan sensitivitas 95% (P(alarm | serangan) = 0,95) dan false positive rate 5% (P(alarm | normal) = 0,05). Hanya 1% lalu lintas jaringan berupa serangan. Bila alarm berbunyi, berapa peluang benar-benar ada serangan? Amati juga pengaruh base rate.

In [ ]:
def posterior(prior, tpr=0.95, fpr=0.05):
    return tpr*prior / (tpr*prior + fpr*(1-prior))
prior = 0.01
print(f"P(alarm) = {0.95*prior + 0.05*(1-prior):.4f}")
print(f"P(serangan | alarm) = {posterior(prior):.4f}")
sim_serangan = np.random.rand(1_000_000) < prior
alarm = np.where(sim_serangan, np.random.rand(1_000_000) < 0.95, np.random.rand(1_000_000) < 0.05)
print(f"Simulasi            = {sim_serangan[alarm].mean():.4f}")
for pr in [0.001, 0.01, 0.1, 0.5]:
    print(f"base rate {pr:5.3f} -> P(serangan | alarm) = {posterior(pr):.4f}")
pr = np.linspace(0.001, 0.5, 200)
plt.plot(pr, posterior(pr)); plt.xlabel("Base rate serangan"); plt.ylabel("P(serangan | alarm)")
plt.title("Pengaruh base rate pada posterior (TPR=0.95, FPR=0.05)"); simpan("A16_ids")

**Interpretasi:** P(serangan|alarm) hanya 0,161 karena serangan jarang (base rate 1%).

## 7. Permutasi dan kombinasi

### A12. Permutasi dan kombinasi: memilih 2 dari 4 server

Dari 4 server A, B, C, D dipilih 2 server (mis. untuk replikasi data). Hitung banyak cara untuk empat skenario: (a) berurutan dengan pengembalian, (b) berurutan tanpa pengembalian, (c) tak berurutan tanpa pengembalian, (d) tak berurutan dengan pengembalian. Verifikasi rumus dengan enumerasi `itertools`, dan uji pendekatan Stirling untuk $n!$.

In [ ]:
st = "ABCD"; n, r = 4, 2
hasil = {
    "Urut, dengan pengembalian (n^r)":          (n**r,                len(list(itertools.product(st, repeat=r)))),
    "Urut, tanpa pengembalian (permutasi)":     (math.perm(n, r),     len(list(itertools.permutations(st, r)))),
    "Tak urut, tanpa pengembalian (kombinasi)": (math.comb(n, r),     len(list(itertools.combinations(st, r)))),
    "Tak urut, dengan pengembalian":            (math.comb(n+r-1, r), len(list(itertools.combinations_with_replacement(st, r)))),
}
for k, (rumus, enum) in hasil.items():
    print(f"{k:45s} rumus={rumus:3d} | enumerasi={enum:3d}")
print("(slide: 16, 12, 6, 10)")
for k in [5, 10, 20]:
    s = math.sqrt(2*math.pi) * math.exp(-k) * k**(k+0.5)
    print(f"{k}! = {math.factorial(k)} | Stirling = {s:.4g} | galat = {abs(s-math.factorial(k))/math.factorial(k)*100:.2f}%")

**Interpretasi:** Rumus = enumerasi = slide (16, 12, 6, 10); galat Stirling mengecil seiring n membesar.

### A13. Kombinatorik: PIN dan pemilihan tim

(a) PIN 4 digit tanpa pengulangan digit ditebak acak. Berapa peluang tebakan benar? (b) Dari 10 mahasiswa (6 Informatika, 4 Sistem Informasi) dipilih tim 3 orang secara acak. Berapa peluang semuanya dari Informatika? Verifikasi dengan enumerasi.

In [ ]:
ruang = math.perm(10, 4)
print("Jumlah PIN tanpa pengulangan:", ruang, "-> P =", 1/ruang)
p = math.comb(6, 3) / math.comb(10, 3)
print("P(tim semua Informatika) =", Fraction(math.comb(6,3), math.comb(10,3)), "=", round(p, 4))
tim = list(itertools.combinations(range(10), 3))
ok = sum(1 for t in tim if all(i < 6 for i in t))
print("Enumerasi itertools:", ok, "/", len(tim), "=", round(ok/len(tim), 4))

**Interpretasi:** P(PIN benar)=1/5040; P(tim semua Informatika)=1/6, cocok dengan enumerasi.

## 8. Eksplorasi tambahan: simulasi dan aplikasi

### A14. Simulasi Monte Carlo: tabrakan pada tabel hash kecil (Birthday Paradox)

Sebanyak $n$ kunci di-hash secara acak seragam ke tabel hash berukuran 365 slot. Berapa peluang minimal dua kunci jatuh pada slot yang sama? Hitung untuk n = 10, 23, 40, bandingkan teori dan simulasi, dan cari n terkecil sehingga peluangnya ≥ 0,5.

In [ ]:
def p_bday(n): return 1 - math.prod((365-i)/365 for i in range(n))
def sim_bday(n, T=20000):
    return np.mean([len(set(np.random.randint(0, 365, n))) < n for _ in range(T)])
for n in [10, 23, 40]:
    print(f"n={n}: teori={p_bday(n):.4f}, simulasi={sim_bday(n):.4f}")
print("n terkecil dengan P >= 0.5:", next(n for n in range(1, 366) if p_bday(n) >= 0.5))
ns = range(1, 80)
plt.plot(ns, [p_bday(n) for n in ns]); plt.axhline(0.5, ls="--", c="r")
plt.xlabel("Jumlah kunci"); plt.ylabel("P(ada tabrakan)"); plt.title("Tabrakan hash (365 slot)")
simpan("A11_birthday")

**Interpretasi:** Cukup 23 kunci untuk peluang tabrakan 0,5073, padahal tabel punya 365 slot; teori ≈ simulasi.

### A15. Simulasi: Monty Hall (3 server)

Salah satu dari 3 server menyimpan file backup yang dicari. Anda menebak satu server; administrator yang tahu isi server membuka satu server lain yang kosong, lalu menawarkan Anda mengganti tebakan. Berapa peluang menemukan file bila tetap dan bila ganti?

In [ ]:
def monty(ganti, T=100_000):
    backup = np.random.randint(0, 3, T); tebak = np.random.randint(0, 3, T)
    if not ganti: return (backup == tebak).mean()
    return (backup != tebak).mean()
print("Tetap  :", round(monty(False), 4), "(teori 1/3)")
print("Ganti  :", round(monty(True), 4), "(teori 2/3)")
p_ganti = monty(True)
lo, hi = proportion_confint(int(p_ganti*100_000), 100_000, method="wilson")
print(f"statsmodels: CI 95% strategi ganti = ({lo:.4f}, {hi:.4f}) | memuat 2/3? {lo <= 2/3 <= hi}")

**Interpretasi:** Ganti tebakan berhasil ≈ 2/3, tetap ≈ 1/3. CI 95% simulasi (statsmodels) memuat 2/3.

### A16. Law of Large Numbers: generator bit acak

Sebuah generator bit acak yang adil (bit 0 dan 1 sama mungkin) menghasilkan 5000 bit. Amati bagaimana proporsi bit 1 berubah terhadap banyaknya bit yang dibangkitkan.

In [ ]:
bits = np.random.randint(0, 2, 5000)
rata = np.cumsum(bits) / np.arange(1, 5001)
for n in [10, 100, 1000, 5000]:
    print(f"Setelah {n:5d} bit: proporsi bit 1 = {rata[n-1]:.4f} | selisih dari 0.5 = {abs(rata[n-1]-0.5):.4f}")
plt.plot(rata); plt.axhline(0.5, c="r", ls="--")
plt.xlabel("Jumlah bit"); plt.ylabel("Proporsi bit 1"); plt.title("Law of Large Numbers")
simpan("A14_lln")

**Interpretasi:** Proporsi bit 1 mendekati 0,5 seiring bertambahnya bit yang dibangkitkan (Law of Large Numbers).

### A17. Tabrakan hash: generalisasi A14

Sebanyak $n=1000$ kunci di-hash secara acak seragam ke $m=10^6$ bucket. Berapa peluang terjadi minimal satu tabrakan (dua kunci pada bucket yang sama)? Bandingkan nilai eksak, pendekatan $1-e^{-n(n-1)/2m}$, dan simulasi. Berapa $n$ agar peluang tabrakan mencapai 50%?

In [ ]:
m, n = 10**6, 1000
eksak = 1 - math.prod((m-i)/m for i in range(n))
aprox = 1 - math.exp(-n*(n-1)/(2*m))
sim = np.mean([len(set(np.random.randint(0, m, n))) < n for _ in range(3000)])
print(f"Eksak = {eksak:.4f} | Pendekatan = {aprox:.4f} | Simulasi = {sim:.4f}")
print("n untuk P=0.5 (~ sqrt(2 m ln2)) =", round(math.sqrt(2*m*math.log(2))))

**Interpretasi:** P(tabrakan)=0,3933 (eksak) ≈ pendekatan 0,3932 ≈ simulasi; 50% tercapai pada ±1177 kunci.

### A18. Probabilitas simbolik dengan sympy: reliabilitas sistem

Tiap komponen berfungsi dengan peluang $p$ dan saling bebas. Turunkan reliabilitas sistem (a) 2 komponen seri, (b) 2 komponen paralel, (c) komponen A seri dengan paralel (B, C). Hitung untuk p = 0,9, dan cari $p$ minimum agar sistem paralel mencapai reliabilitas 0,999.

In [ ]:
p = sp.symbols("p", positive=True)
R_seri = p**2
R_paralel = sp.expand(1 - (1-p)**2)
R_campur = sp.expand(p * (1 - (1-p)**2))
print("Seri    :", R_seri, "| p=0.9 ->", R_seri.subs(p, 0.9))
print("Paralel :", R_paralel, "| p=0.9 ->", R_paralel.subs(p, 0.9))
print("Campur  :", R_campur, "| p=0.9 ->", R_campur.subs(p, 0.9))
akar = sp.solve(sp.Eq(R_paralel, sp.Rational(999, 1000)), p)
print("p minimum untuk paralel >= 0.999:", [round(float(a), 4) for a in akar if a <= 1])

**Interpretasi:** Untuk p=0,9: seri 0,81; paralel 0,99; campuran 0,891. Paralel ≥ 0,999 perlu p ≥ 0,9684.

### A19. Rantai Markov sederhana: status server

Status server tiap jam adalah Up atau Down. Bila Up, jam berikutnya Up dengan peluang 0,95; bila Down, jam berikutnya kembali Up dengan peluang 0,40. Hitung (a) peluang Down dua jam lagi bila sekarang Up, (b) distribusi stasioner (proporsi waktu jangka panjang). Verifikasi dengan simulasi.

In [ ]:
P = np.array([[0.95, 0.05],
              [0.40, 0.60]])        # baris/kolom: 0 = Up, 1 = Down
P2 = np.linalg.matrix_power(P, 2)
print("P(Down 2 jam lagi | sekarang Up) =", round(P2[0, 1], 4), "(manual 0.95*0.05 + 0.05*0.60 =", round(0.95*0.05 + 0.05*0.60, 4), ")")
vals, vecs = np.linalg.eig(P.T)
pi = np.real(vecs[:, np.argmax(np.real(vals))]); pi = pi / pi.sum()
print("Distribusi stasioner (eigen)  :", pi.round(4), "| manual (8/9, 1/9) =", round(8/9, 4), round(1/9, 4))
print("Baris P^50 (konvergen)        :", np.linalg.matrix_power(P, 50)[0].round(4))
state, hitung = 0, np.zeros(2)
for _ in range(200_000):
    state = 0 if np.random.rand() < P[state, 0] else 1
    hitung[state] += 1
print("Simulasi proporsi Up/Down     :", (hitung/hitung.sum()).round(4))
sns.heatmap(P, annot=True, fmt=".2f", cmap="Blues", xticklabels=["Up", "Down"], yticklabels=["Up", "Down"])
plt.title("Matriks transisi status server"); simpan("A17_markov")

**Interpretasi:** P(Down 2 jam lagi)=0,0775; distribusi stasioner (8/9, 1/9) ≈ simulasi.

---
# BAGIAN B: DISTRIBUSI PROBABILITAS

## 1. Distribusi Hipergeometrik

### B1. Hipergeometrik: audit server rusak

Dari 12 server, 2 rusak. Dipilih 6 server tanpa pengembalian untuk diaudit. Hitung distribusi jumlah server rusak yang terpilih, nilai harapan, dan varians.

In [ ]:
Np, k, n = 12, 2, 6
d = stats.hypergeom(M=Np, n=k, N=n)       # scipy: M=populasi, n=sukses populasi, N=ukuran sampel
for x in [2, 1, 0]:
    manual = math.comb(k, x) * math.comb(Np-k, n-x) / math.comb(Np, n)
    print(f"f({x};12,6,2): manual={manual:.4f} | scipy={d.pmf(x):.4f}")
print("(slide: 0.2273, 0.5454, 0.2273)")
print("sympy eksak:", [sp.Rational(math.comb(k, x)*math.comb(Np-k, n-x), math.comb(Np, n)) for x in [2, 1, 0]], "(6/11 = 0.545454..., slide memotong jadi 0.5454)")
print("E(X) = nk/N =", n*k/Np, "| scipy:", d.mean())
var_manual = n*k*(Np-k)*(Np-n) / (Np**2 * (Np-1))
print("Var manual =", round(var_manual, 4), "| scipy:", round(d.var(), 4))
x = np.arange(0, 3); plt.bar(x, d.pmf(x), color="tab:purple")
plt.xticks(x); plt.xlabel("Server rusak terpilih"); plt.title("Hipergeometrik N=12, k=2, n=6")
simpan("B1_hipergeometrik")

**Interpretasi:** f(0)=0,2273; f(1)=0,5455; f(2)=0,2273; E=1; Var=0,4545 (slide membulatkan 0,5454). `sympy` menunjukkan nilai eksak 5/22, 6/11, 5/22.

### B2. Hipergeometrik: verifikasi file backup

Dari 50 file backup terdapat 5 file korup. Diambil 8 file tanpa pengembalian untuk diverifikasi checksum. Hitung P(tepat 1 korup), E, Var, dan bandingkan dengan Binomial (n=8, p=0,1) untuk melihat efek pengambilan tanpa pengembalian.

In [ ]:
M, K, n = 50, 5, 8
d = stats.hypergeom(M, K, n)
print("P(X=1) =", round(d.pmf(1), 4), "| manual:", round(math.comb(5,1)*math.comb(45,7)/math.comb(50,8), 4))
print("E =", round(d.mean(), 4), "Var =", round(d.var(), 4))
b = stats.binom(n, K/M)
print("Binomial pembanding: P(X=1) =", round(b.pmf(1), 4), "| Var =", round(b.var(), 4))
print("Faktor koreksi (N-n)/(N-1) =", round((M-n)/(M-1), 4), "-> Var hiper =", round(b.var()*(M-n)/(M-1), 4))

**Interpretasi:** P(X=1)=0,4226; variansnya lebih kecil dari Binomial (0,6171 vs 0,72) karena tanpa pengembalian.

### B3. Hipergeometrik: code review

Dari 30 pull request, 6 mengandung bug. Reviewer memeriksa 5 PR secara acak. Hitung P(minimal 1 bug ditemukan), E, dan banyak PR minimal yang harus diperiksa agar P(minimal 1 bug) ≥ 0,9. Verifikasi dengan simulasi.

In [ ]:
M, K, n = 30, 6, 5
d = stats.hypergeom(M, K, n)
manual = 1 - math.comb(M-K, n) / math.comb(M, n)
print("P(X>=1) scipy =", round(d.sf(0), 4), "| manual =", round(manual, 4))
print("E =", d.mean(), "| Var =", round(d.var(), 4))
sim = np.random.hypergeometric(K, M-K, n, N)
print("Simulasi P(X>=1) =", round((sim >= 1).mean(), 4))
lo, hi = proportion_confint(int((sim >= 1).sum()), N, method="wilson")
print(f"statsmodels: CI 95% simulasi = ({lo:.4f}, {hi:.4f}) | manual di dalam CI? {lo <= manual <= hi}")
n_min = next(k for k in range(1, M+1) if stats.hypergeom(M, K, k).sf(0) >= 0.9)
print("PR minimal diperiksa agar P>=0.9:", n_min)
x = np.arange(0, 6); plt.bar(x, d.pmf(x), color="tab:purple"); plt.xlabel("Bug ditemukan")
plt.title("Hipergeometrik N=30, K=6, n=5"); simpan("H3_codereview")

**Interpretasi:** P(minimal 1 bug ditemukan)=0,7017 (E=1 bug); perlu memeriksa minimal 9 PR agar peluangnya ≥ 0,9. CI 95% simulasi (statsmodels) memuat nilai manual.

## 2. Distribusi Binomial

### B4. Binomial: test case gagal pada CI pipeline

Pipeline CI menjalankan 5 test case; masing-masing gagal dengan peluang 0,25 secara independen. Hitung P(tepat 3 test gagal), tabel distribusi lengkap, E, Var, dan kemencengan. Amati pengaruh $n$ pada bentuk distribusi.

In [ ]:
n, p = 5, 0.25
d = stats.binom(n, p)
print("f(3;5,0.25) =", round(d.pmf(3), 4), "(slide: 0.0879)")
tb = pd.DataFrame({"gagal": range(6), "cara": [math.comb(5, i) for i in range(6)],
                   "prob": d.pmf(range(6)).round(4)})
print(tb.to_string(index=False), "\nJumlah =", tb.prob.sum().round(4))
q = 1-p
print("E =", d.mean(), "| Var =", d.var(), "| skewness =", round((q-p)/math.sqrt(n*p*q), 4), "(q>p -> miring kanan)")
x_, p_ = sp.symbols("x p", positive=True)
pmf_s = sp.binomial(5, x_) * p_**x_ * (1-p_)**(5-x_)
E_s = sp.simplify(sum(i*pmf_s.subs(x_, i) for i in range(6)))
V_s = sp.simplify(sum(i**2*pmf_s.subs(x_, i) for i in range(6)) - E_s**2)
print("sympy: E =", sp.factor(E_s), "| Var =", sp.factor(V_s), "| p=1/4 ->", E_s.subs(p_, sp.Rational(1, 4)), V_s.subs(p_, sp.Rational(1, 4)))
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
for a, nn in zip(ax, [5, 20, 50]):
    xx = np.arange(0, nn+1); a.bar(xx, stats.binom(nn, .25).pmf(xx)); a.set_title(f"n={nn}, p=0.25")
simpan("B2_binomial_n")

**Interpretasi:** P(3 test gagal)=0,0879; E=1,25; Var=0,9375; bentuk makin simetris bila n besar. `sympy` menurunkan E = 5p dan Var = 5p(1−p).

### B5. Binomial: paket data rusak

Dari 20 paket data, tiap paket rusak dengan peluang 0,1 secara independen. Hitung P(tepat 2 rusak), P(≤ 3 rusak), P(minimal 1 rusak), E dan Var, lalu verifikasi dengan simulasi.

In [ ]:
n, p = 20, 0.1
d = stats.binom(n, p)
print("P(X=2) =", round(d.pmf(2), 4), "| manual:", round(math.comb(20,2)*0.1**2*0.9**18, 4))
print("P(X<=3) =", round(d.cdf(3), 4), "| P(X>=1) =", round(1-d.pmf(0), 4))
print("E[X] =", d.mean(), "(np) | Var =", d.var(), "(npq)")
x = np.arange(0, 11); plt.bar(x, d.pmf(x))
plt.title("Binomial n=20, p=0.1"); plt.xlabel("Paket rusak"); simpan("B9_binomial")
sim = d.rvs(100000)
print("Simulasi: mean =", round(sim.mean(), 4), "| var =", round(sim.var(), 4), "| P(X=2) =", round((sim == 2).mean(), 4))

**Interpretasi:** P(X=2)=0,2852; P(X≥1)=0,8784; E=2; Var=1,8.

### B6. Binomial: keberhasilan request dan kuantil

Dari 10 request, tiap request sukses dengan peluang 0,9 (independen). Hitung P(minimal 8 sukses), P(semua sukses), dan jumlah sukses terbesar k sehingga P(X ≥ k) ≥ 0,95.

In [ ]:
n, p = 10, 0.9
d = stats.binom(n, p)
print("P(X>=8) =", round(d.sf(7), 4), "| P(X=10) =", round(d.pmf(10), 4), "| manual 0.9^10 =", round(0.9**10, 4))
k_min = max(k for k in range(n+1) if d.sf(k-1) >= 0.95)
print("k terbesar dengan P(X>=k)>=0.95:", k_min, "| P =", round(d.sf(k_min-1), 4))
print("E =", d.mean(), "| Var =", round(d.var(), 4))
x = np.arange(0, 11); plt.bar(x, d.pmf(x), color=["tab:blue" if i < 8 else "tab:green" for i in x])
plt.xlabel("Request sukses"); plt.title("Binomial n=10, p=0.9 (hijau: X>=8)"); simpan("Bi3_request")

**Interpretasi:** P(X≥8)=0,9298; P(semua sukses)=0,3487. Dengan peluang ≥ 95%, minimal 7 request sukses (P(X≥7)=0,9872).

## 3. Distribusi Poisson

### B7. Poisson: bug per modul kode

Rata-rata ditemukan 2 bug per modul kode. Hitung P(tepat 1 bug), P(0 bug), P(≥ 3 bug), serta tunjukkan bahwa Poisson adalah limit Binomial bila $np=\lambda$ tetap.

In [ ]:
lam = 2
d = stats.poisson(lam)
print("f(1;2) manual =", round(lam*math.exp(-lam), 4), "| scipy =", round(d.pmf(1), 4), "(slide: 0.2707)")
print("P(X=0) =", round(d.pmf(0), 4), "| P(X>=3) =", round(d.sf(2), 4))
print("E = Var =", d.mean(), d.var(), "| skewness = 1/sqrt(lam) =", round(1/math.sqrt(lam), 4))
lam_s, k_s = sp.symbols("lambda k", positive=True)
pmf_p = sp.exp(-lam_s) * lam_s**k_s / sp.factorial(k_s)
E_p = sp.simplify(sp.summation(k_s*pmf_p, (k_s, 0, sp.oo)))
V_p = sp.simplify(sp.summation(k_s**2*pmf_p, (k_s, 0, sp.oo)) - E_p**2)
print("sympy: E =", E_p, "| Var =", V_p)
x = np.arange(0, 10); plt.bar(x, d.pmf(x), color="tab:orange")
plt.title("Poisson λ=2 (bug per modul)"); simpan("B3_poisson")
for nn in [10, 100, 1000]:
    print(f"Binom(n={nn}, p={lam/nn}) P(X=1) = {stats.binom(nn, lam/nn).pmf(1):.4f}")

**Interpretasi:** P(X=1)=0,2707; E=Var=2; Binomial(n,2/n) menuju Poisson bila n besar. `sympy` membuktikan E = Var = λ.

### B8. Poisson: request server

Server menerima rata-rata 4 request per detik. Hitung P(tepat 6 request) dan peluang overload bila kapasitas 8 request/detik (P(X > 8)).

In [ ]:
lam = 4
d = stats.poisson(lam)
print("P(X=6) =", round(d.pmf(6), 4), "| manual:", round(math.exp(-4)*4**6/math.factorial(6), 4))
print("P(X>8) =", round(d.sf(8), 4), "(overload)")
print("E =", d.mean(), "Var =", d.var())
x = np.arange(0, 15); plt.bar(x, d.pmf(x), color="tab:orange")
plt.title("Poisson λ=4"); simpan("B10_poisson")
print("Simulasi P(X>8) =", round((d.rvs(500_000) > 8).mean(), 4))

**Interpretasi:** P(X=6)=0,1042; peluang overload P(X>8)=0,0214.

### B9. Poisson: penjumlahan dua proses independen

Dua server independen menerima rata-rata 3 dan 5 request per detik. Tunjukkan bahwa total request per detik berdistribusi Poisson(8): bandingkan P(total = 8) dari konvolusi, dari Poisson(8), dan dari simulasi.

In [ ]:
l1, l2, k = 3, 5, 8
konv = sum(stats.poisson(l1).pmf(i) * stats.poisson(l2).pmf(k-i) for i in range(k+1))
print("Konvolusi P(S=8) =", round(konv, 4), "| Poisson(8) =", round(stats.poisson(l1+l2).pmf(k), 4))
S = stats.poisson(l1).rvs(N) + stats.poisson(l2).rvs(N)
print("Simulasi: P(S=8) =", round((S == k).mean(), 4), "| mean =", round(S.mean(), 3), "| var =", round(S.var(), 3))
x = np.arange(0, 20)
sns.histplot(S, discrete=True, stat="probability", alpha=.6, label="Simulasi")
plt.plot(x, stats.poisson(8).pmf(x), "ro-", label="Poisson(8)"); plt.legend()
plt.title("Penjumlahan dua Poisson"); simpan("P3_penjumlahan")

**Interpretasi:** Konvolusi = Poisson(8) = 0,1396 dan simulasi cocok: jumlah Poisson independen tetap Poisson dengan λ = λ₁ + λ₂.

## 4. Distribusi Normal

### B10. Distribusi Normal: waktu eksekusi query dan tabel z

Waktu eksekusi sebuah query (ms) berdistribusi $X\sim N(12,3^2)$. Hitung P(X<15), P(X<9), P(9<X<15), nilai $x$ dengan P(X<x)=0,8, serta beberapa peluang pada normal baku dan aturan empiris 68–95–99,7.

In [ ]:
mu, sd = 12, 3
d = stats.norm(mu, sd)
print("P(X<15)    =", round(d.cdf(15), 4), "(slide: 0.8413)")
print("P(X<9)     =", round(d.cdf(9), 4))
print("P(9<X<15)  =", round(d.cdf(15)-d.cdf(9), 4))
print("x utk P=0.8 =", round(d.ppf(0.8), 4), "(slide: 14.5249)")
z = stats.norm()
print("P(Z<3)     =", round(z.cdf(3), 4), "(slide: 0.9987)")
print("P(0<Z<3)   =", round(z.cdf(3)-0.5, 4), "(slide: 0.4987)")
print("P(1<Z<3)   =", round(z.cdf(3)-z.cdf(1), 4))
print("P(Z>1.5)   =", round(z.sf(1.5), 4))
print("z utk P=0.65 =", round(z.ppf(0.65), 4), "(slide menulis 0.3843; hitungan Python 0.3853)")
print("Aturan empiris: P(|Z|<1) =", round(z.cdf(1)-z.cdf(-1), 4),
      "| P(|Z|<2) =", round(z.cdf(2)-z.cdf(-2), 4), "| P(|Z|<3) =", round(z.cdf(3)-z.cdf(-3), 4))
t_ = sp.symbols("t", real=True)
luas = sp.integrate(sp.exp(-t_**2/2)/sp.sqrt(2*sp.pi), (t_, -1, 1))
print("sympy: P(-1<Z<1) =", sp.simplify(luas), "=", round(float(luas), 4))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
x = np.linspace(0, 24, 400)
ax[0].plot(x, d.pdf(x)); m = (x > 9) & (x < 15); ax[0].fill_between(x[m], d.pdf(x[m]), alpha=.3)
ax[0].set_title("pdf N(12, 3²): area P(9<X<15)")
ax[1].plot(x, d.cdf(x)); ax[1].set_title("cdf N(12, 3²)")
simpan("B4_normal")

**Interpretasi:** P(X<15)=0,8413; x untuk P=0,8 adalah 14,5249 ms. Python memberi z=0,3853, slide menulis 0,3843. `sympy`: P(−1<Z<1) = erf(√2/2) = 0,6827.

### B11. Normal: waktu respons dan SLA

Waktu respons API berdistribusi Normal dengan μ = 200 ms dan σ = 30 ms. Hitung P(X < 150), P(180 < X < 250), persentil ke-95 (batas SLA), dan skor-z untuk X = 250.

In [ ]:
mu, sigma = 200, 30
d = stats.norm(mu, sigma)
print("P(X<150) =", round(d.cdf(150), 4))
print("P(180<X<250) =", round(d.cdf(250)-d.cdf(180), 4))
print("Persentil 95 =", round(d.ppf(0.95), 2), "ms (SLA)")
print("Z untuk X=250:", round((250-mu)/sigma, 3))
nd = statistics.NormalDist(mu, sigma)
print("statistics.NormalDist: P(X<150) =", round(nd.cdf(150), 4), "| persentil 95 =", round(nd.inv_cdf(0.95), 2))
x = np.linspace(100, 300, 300); plt.plot(x, d.pdf(x))
plt.fill_between(x[(x>=180)&(x<=250)], d.pdf(x[(x>=180)&(x<=250)]), alpha=.3)
plt.title("Normal μ=200, σ=30"); simpan("B17_normal")

**Interpretasi:** P(X<150)=0,0478; P(180<X<250)=0,6997; persentil ke-95 = 249,35 ms. `statistics.NormalDist` memberi hasil sama dengan scipy.

### B12. Pendekatan Normal terhadap Binomial: cache miss

Dari $n$ akses memori, tiap akses cache miss dengan peluang 0,25 (independen), sehingga jumlah miss $X\sim$ Binomial(n, 0,25). Untuk n = 20 dan 50, bandingkan P(X ≤ np+2) dari Binomial eksak dengan pendekatan Normal memakai koreksi kontinuitas 0,5.

In [ ]:
for nn in [20, 50]:
    b = stats.binom(nn, 0.25); nm = stats.norm(nn*.25, math.sqrt(nn*.25*.75))
    batas = int(nn*.25) + 2
    print(f"n={nn}: P(X<={batas}) binomial={b.cdf(batas):.4f} | normal tanpa koreksi={nm.cdf(batas):.4f} | normal (koreksi 0.5)={nm.cdf(batas+0.5):.4f}")
nn = 50; xx = np.arange(0, 31)
plt.bar(xx, stats.binom(nn, .25).pmf(xx), alpha=.6, label="Binomial")
xc = np.linspace(0, 30, 300); plt.plot(xc, stats.norm(nn*.25, math.sqrt(nn*.25*.75)).pdf(xc), "r", label="Normal")
plt.legend(); plt.title("Binomial n=50, p=0.25 ≈ Normal"); simpan("B5_binom_normal")

**Interpretasi:** Normal dengan koreksi kontinuitas jauh lebih dekat ke Binomial eksak daripada tanpa koreksi.

### B13. Pendekatan Normal: Central Limit Theorem pada data Poisson

Jumlah request per detik ~ Poisson(4). Ambil 10.000 sampel berukuran n = 30 dan hitung rata-ratanya. Apakah rata-rata sampel mendekati Normal(4, 4/30)? Periksa mean, SD, kemencengan, dan Q-Q plot.

In [ ]:
lam = 4
sampel = np.random.poisson(lam, size=(10000, 30)).mean(axis=1)
print("Mean:", round(sampel.mean(), 3), "(teori 4) | SD:", round(sampel.std(), 3), "(teori", round(math.sqrt(lam/30), 3), ")")
for n in [1, 5, 30]:
    s = np.random.poisson(lam, size=(10000, n)).mean(axis=1)
    print(f"n={n:2d}: skewness = {stats.skew(s):.3f} | teori 1/sqrt(n*lam) = {1/math.sqrt(n*lam):.3f}")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
sns.histplot(sampel, bins=30, stat="density", kde=True, ax=ax[0])
x = np.linspace(sampel.min(), sampel.max(), 200)
ax[0].plot(x, stats.norm(lam, math.sqrt(lam/30)).pdf(x), "r"); ax[0].set_title("Rata-rata 30 sampel Poisson(4)")
sm.qqplot(sampel, line="s", ax=ax[1]); ax[1].set_title("Q-Q plot terhadap Normal")
simpan("CLT_poisson")

**Interpretasi:** Mean 4,0 dan SD 0,366 ≈ teori 0,365; kemencengan menurun ke 0 dan Q-Q plot hampir lurus, jadi rata-rata sampel mendekati Normal (CLT).

## 5. Ringkasan

### B14. Ringkasan empat distribusi (pandas)

Rangkum nilai harapan dan varians dari distribusi yang dipakai dalam notebook ini.

In [ ]:
dists = {
    "Hipergeometrik(12,2,6)":  stats.hypergeom(12, 2, 6),
    "Hipergeometrik(30,6,5)":  stats.hypergeom(30, 6, 5),
    "Binomial(5,0.25)":        stats.binom(5, .25),
    "Binomial(20,0.1)":        stats.binom(20, .1),
    "Poisson(2)":              stats.poisson(2),
    "Poisson(4)":              stats.poisson(4),
    "Normal(12,3)":            stats.norm(12, 3),
    "Normal(200,30)":          stats.norm(200, 30),
}
tabel = pd.DataFrame({"Distribusi": list(dists), "E[X]": [d.mean() for d in dists.values()],
                      "Var(X)": [d.var() for d in dists.values()]}).round(4)
print(tabel.to_string(index=False))

**Interpretasi:** Poisson: E = Var; Binomial: Var < E; Hipergeometrik: Var lebih kecil dari Binomial (tanpa pengembalian); Normal ditentukan oleh μ dan σ².

---
## Kesimpulan
- Hasil perhitungan manual cocok dengan `scipy.stats`, `itertools`, dan simulasi; tiga selisih dengan slide dicatat (probabilitas total, Hipergeometrik, tabel z Normal).
- Teorema Bayes dan base rate penting dalam Informatika (filter spam, IDS), dan simulasi Monte Carlo memverifikasi teori.
- Hipergeometrik dipakai untuk sampling tanpa pengembalian, Binomial untuk percobaan sukses-gagal, Poisson untuk kejadian per satuan waktu, dan Normal untuk pendekatan (Binomial, CLT).